# Deploying the Car Price API to AWS (Docker → ECR → ECS Fargate)

A step-by-step guide to containerising the Flask API in this repository, verifying it on
your own machine, and running it on **AWS ECS Fargate** — written to be followed with no
prior Docker or AWS experience.

**Scope of this notebook**

- Build a Docker **image** of the API and confirm it works **locally** (with real output).
- Configure the AWS CLI and confirm the **correct account/Region** are active.
- Push the image to **Amazon ECR** and run it on **Amazon ECS (Fargate)**.
- Test the live endpoint, then **remove every resource** so it stops incurring cost.

> **Note.** The local Docker steps below were executed on a real machine and show their
> actual output. The AWS steps are given for the **AWS Console** (point-and-click), with the
> equivalent **AWS CLI** commands beside them.

## 1. Overview

Four concepts, defined plainly:

| Term | Definition |
| :--- | :--------- |
| **Docker image** | A self-contained snapshot of the application together with everything it needs to run (Python, libraries, the trained model files). It behaves identically on any machine. |
| **Docker container** | A running instance of an image. One image can start many identical containers. |
| **Amazon ECR** (Elastic Container Registry) | A private registry that stores your images inside AWS — conceptually a private Docker Hub. |
| **Amazon ECS + Fargate** | The service that runs your container in AWS. **Fargate** is the *serverless* option: AWS provisions and manages the compute, and you pay only while the container runs. |

**The end-to-end flow:**

```
  Local machine                              AWS
  ┌──────────────┐  docker push  ┌───────┐  pulls & runs  ┌───────────────┐
  │ Docker image │ ─────────────►│  ECR  │ ──────────────►│ ECS (Fargate) │──► http://<public-ip>:5000
  └──────────────┘               └───────┘                └───────────────┘
```

> **Key principle.** If the container runs correctly on your machine, it runs identically on
> AWS, because the image carries its entire runtime with it. That is why we validate locally
> (sections 3–4) before deploying.

## 2. Prerequisites

| Requirement | How to confirm / obtain |
| :---------- | :---------------------- |
| **Docker Desktop** installed and running | Launch Docker Desktop; verify with `docker version`. |
| **An AWS account** with Console access | You sign in to the web Console at [console.aws.amazon.com](https://console.aws.amazon.com). |
| **AWS CLI** *(for the command-line path)* | `aws --version`. Optional if you use the Console or CloudShell. |
| **This repository**, checked out locally | It contains the `Dockerfile`, `app.py`, and the trained `models/`. |

> **Choose one AWS Region and use it throughout** (for example `us-east-1`). ECR
> repositories, ECS clusters, and logs are **Region-scoped**; mixing Regions is a common
> source of "resource not found" errors. The Region selector is in the top-right of the Console.

## 3. Build the Docker image

A **Dockerfile** is the build recipe. This repository provides two:

- **`Dockerfile`** — production image, served by **gunicorn**. **Use this one for AWS.**
- **`Dockerfile.dev`** — Flask's built-in development server, for local experimentation only.

Build the production image (`-t` names it; `.` points to the recipe in the current folder).
Run these cells from the repository root; in a notebook the `!` prefix runs a shell command.

In [1]:
!docker build -t car-price-api .

#8 [4/6] RUN pip install --no-cache-dir -r requirements.txt
#8 CACHED
#9 [5/6] COPY app.py car_model.py wsgi.py ./
#9 DONE 0.0s
#10 [6/6] COPY models/ ./models/
#10 DONE 0.1s
#11 exporting to image
#11 naming to docker.io/library/car-price-api:latest done
#11 DONE 0.5s


The first build takes a few minutes (it downloads Python and installs pandas / scikit-learn);
later builds take seconds because Docker caches unchanged steps (`CACHED` above). Confirm the
image exists and note its size:

In [2]:
!docker images car-price-api

REPOSITORY      TAG       IMAGE ID       CREATED          SIZE
car-price-api   latest    c3d83f1fef3b   2 minutes ago    728MB


## 4. Run and test the container locally

Validate the image on your machine before deploying. `-d` runs it in the background,
`-p 5000:5000` maps your machine's port 5000 to the container's, and `--name` labels it.
`docker run` prints the new container's ID.

In [3]:
!docker run -d -p 5000:5000 --name car-price-api car-price-api

eb9878d925e5437a4cacd6d740dc0b1bb64cacbb86a3aebe54c6e46d9481aef2


Confirm it is running (`Up`, with port 5000 published):

In [4]:
!docker ps

IMAGE           STATUS         PORTS
car-price-api   Up 3 seconds   0.0.0.0:5000->5000/tcp, [::]:5000->5000/tcp


**Health check** — is the service up and are the models loaded? (ECS will use this same endpoint.)

In [5]:
import requests, json

r = requests.get("http://localhost:5000/health")
print(r.status_code)
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

200
{
  "status": "healthy"
}


**Alternative — the same request using `curl`.** `curl` is a command-line HTTP client: the
standard way to call an API straight from a terminal, without Python. A bare `curl <url>` sends
a **GET** request.

```bash
curl http://localhost:5000/health
```

**Request a prediction.** `/predict` accepts a `POST` with a JSON body. Only `make` and
`model` are required; any other field is inferred from that model's typical values.

In [6]:
r = requests.post(
    "http://localhost:5000/predict",
    json={"make": "MARUTI", "model": "SWIFT VDI", "age": 3, "km_driven": 45000},
)
print(r.status_code)
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

200
{
  "input_used": {
    "age": 3.0,
    "engine": 1248.0,
    "fuel": "Diesel",
    "km_driven": 45000.0,
    "make": "MARUTI",
    "max_power": 74.0,
    "mileage": 22.9,
    "model": "SWIFT VDI",
    "seats": "5",
    "seller": "Dealer",
    "transmission": "Manual"
  },
  "predicted_price_display": "₹6.50 Lakhs",
  "predicted_price_lakhs": 6.5,
  "price_range": {
    "display": "₹3.99 Lakhs - ₹6.75 Lakhs",
    "high_lakhs": 6.75,
    "label": "Medium",
    "low_lakhs": 3.99
  }
}


**Alternative — the same request using `curl`:**

```bash
curl -X POST http://localhost:5000/predict \
  -H "Content-Type: application/json" \
  -d '{"make":"MARUTI","model":"SWIFT VDI","age":3,"km_driven":45000}'
```

What each flag does:
- **`-X POST`** — set the HTTP method to `POST`.
- **`-H "Content-Type: application/json"`** — tell the server the body is JSON.
- **`-d '{…}'`** — the JSON request body to send.

> **Windows note.** In PowerShell / CMD the single quotes around the JSON are not parsed the
> same way. Either use the Python `requests` cell above, or save the JSON to a file (e.g.
> `body.json`) and pass it with `-d "@body.json"`.

> **Expected behaviour: `GET /predict` returns HTTP 405.** A web browser issues a `GET`
> request, but `/predict` accepts only `POST`. The `405 Method Not Allowed` response below is
> therefore correct, not a defect:

In [7]:
r = requests.get("http://localhost:5000/predict")   # a browser also issues GET
print(r.status_code)
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

405
{
  "error": "Method not allowed.",
  "hint": "/predict only accepts POST with a JSON body. A browser visiting this URL sends GET, which is why you see this error there."
}


**Alternative — the same request using `curl`.** A plain `curl` sends a `GET` (exactly what a
browser does), so it returns the `405`:

```bash
curl http://localhost:5000/predict
```

Inspect the server logs — note **gunicorn** running two worker processes (the production server):

In [8]:
!docker logs car-price-api

[2026-07-09 13:26:50 +0000] [1] [INFO] Starting gunicorn 22.0.0
[2026-07-09 13:26:50 +0000] [1] [INFO] Listening at: http://0.0.0.0:5000 (1)
[2026-07-09 13:26:50 +0000] [1] [INFO] Using worker: sync
[2026-07-09 13:26:50 +0000] [7] [INFO] Booting worker with pid: 7
[2026-07-09 13:26:50 +0000] [8] [INFO] Booting worker with pid: 8


Stop and remove the local container when finished (the image is retained for the push):

In [9]:
!docker rm -f car-price-api

car-price-api


> The image just validated is exactly what will run on AWS. Proceed to the cloud steps.

## 5. Configure the AWS CLI and confirm the active account

On a machine configured with **more than one AWS account**, always confirm *which account and
Region the CLI is using* before running any command, so you never act on the wrong account.

### Console sign-in vs. CLI credentials

These are two different mechanisms — a frequent point of confusion:

| | **Console** (web) | **AWS CLI** (terminal) |
| :-- | :-- | :-- |
| Authenticates with | **username + password** (+ MFA) | **access keys** — an *Access Key ID* and *Secret Access Key* |
| Cannot | run `docker push` / `aws ecs …` directly | be signed in with a username/password |

A username and password alone do not authorise the CLI. For the CLI you need that account's
**access keys**:

- **Create keys:** Console → **IAM → Users → *(your user)* → Security credentials → Create
  access key → "Command Line Interface (CLI)"**. Copy both values (the secret is shown once).
- **No permission to create keys?** Use **AWS CloudShell** — the `>_` icon in the Console's
  top bar opens a terminal already authenticated as your Console user, with no keys to
  configure. Run the `aws` commands there and skip the profile setup below.

### 5.1 Confirm your current identity

```bash
aws sts get-caller-identity
```
```json
{
  "UserId":  "AIDA................",
  "Account": "123456789012",     // confirm this is the account you intend to use
  "Arn":     "arn:aws:iam::123456789012:user/your-user"
}
```

### 5.2 List and inspect your profiles

Keep each account in its own **named profile** rather than overwriting one set of keys:

```bash
aws configure list-profiles      # every configured profile
aws configure list               # the active profile: masked keys, Region, and their source
```

### 5.3 Create or update a profile

```bash
aws configure --profile work
#   AWS Access Key ID     : <access key id for that account>
#   AWS Secret Access Key : <secret access key>
#   Default region name   : us-east-1
#   Default output format : json
```

Configure one profile per account (for example `work` and `personal`). Credentials are stored
in `~/.aws/credentials` (on Windows, `C:\Users\<you>\.aws\credentials`), one section per profile.

### 5.4 Select the profile to use

Per command (explicit, and hard to get wrong):

```bash
aws sts get-caller-identity --profile work
# add --profile work to the ECR/ECS commands in sections 6–9 as well
```

For an entire terminal session:

```bash
export AWS_PROFILE=work          # macOS / Linux
$env:AWS_PROFILE = "work"        # Windows PowerShell
set AWS_PROFILE=work             # Windows CMD
```

Re-run `aws sts get-caller-identity` afterwards to confirm the active account changed.

### 5.5 Confirm the Region

```bash
aws configure get region                             # the active Region
aws configure set region us-east-1 --profile work    # set it if required
```

> **Pre-deployment check.** Before running the commands in sections 6–7, confirm (1) the
> `Account` from `get-caller-identity` is the intended one, and (2) the Region is correct.
> These few seconds prevent deploying resources into the wrong account.

## 6. Push the image to Amazon ECR

ECR stores the image in AWS so that ECS can pull it. The Console generates the exact commands
for you.

### Using the Console
1. Sign in at [console.aws.amazon.com](https://console.aws.amazon.com); confirm the Region (top-right).
2. Search for **ECR** → open **Elastic Container Registry**.
3. **Create repository** → **Private** → name it **`car-price-api`** → **Create**.
4. Open the repository → **View push commands**.
5. Run the four commands it displays, from this repository's folder. They (1) authenticate
   Docker to ECR, (2) build, (3) tag, and (4) push the image.

### Equivalent CLI commands
Replace `<ACCOUNT_ID>` and `<REGION>` with your values (the Console fills these in for you);
add `--profile work` if you use named profiles.

```bash
# 1) Authenticate Docker to your ECR registry
aws ecr get-login-password --region <REGION> | \
  docker login --username AWS --password-stdin <ACCOUNT_ID>.dkr.ecr.<REGION>.amazonaws.com

# 2) Build (already done in section 3)
docker build -t car-price-api .

# 3) Tag the image with its full ECR address
docker tag car-price-api:latest <ACCOUNT_ID>.dkr.ecr.<REGION>.amazonaws.com/car-price-api:latest

# 4) Push
docker push <ACCOUNT_ID>.dkr.ecr.<REGION>.amazonaws.com/car-price-api:latest
```

> **Building on Apple Silicon (M-series) or other arm64 hosts:** Fargate runs **amd64**. Build
> with `docker build --platform linux/amd64 -t car-price-api .`, or the task will fail to start.
> On Windows/Intel this is already the default.

After the push completes, refresh the ECR repository page; the image appears with the `latest` tag.

## 7. Deploy on Amazon ECS (Fargate)

Terminology, simplest first:
- **Task definition** — the blueprint: which image to run, how much CPU/memory, and which port.
- **Task** — a running instance of a task definition.
- **Cluster** — the logical group the tasks belong to.
- **Fargate** — the launch type meaning AWS supplies and manages the compute.

### Using the Console

**A. Create a cluster**
1. Search **ECS** → **Clusters** → **Create cluster**.
2. Name: **`car-price-cluster`**. Keep **AWS Fargate (serverless)** selected → **Create**.

**B. Create a task definition**
1. **Task definitions** → **Create new task definition**.
2. Family: **`car-price-api`**; launch type: **AWS Fargate**.
3. CPU **0.5 vCPU**, Memory **1 GB**.
4. Container:
   - Name: **`car-price-api`**
   - **Image URI:** `<ACCOUNT_ID>.dkr.ecr.<REGION>.amazonaws.com/car-price-api:latest`
   - **Port mappings:** container port **5000**, protocol **TCP**.
5. Enable **log collection** (CloudWatch) so container logs are available.
6. **Create.**

**C. Run the task**
1. Open the cluster → **Tasks** → **Run new task** (or create a **Service** for an always-on deployment).
2. Launch type **Fargate**; task definition **`car-price-api`** (latest revision).
3. Networking:
   - Select your **VPC** and a **public subnet**.
   - **Auto-assign public IP: ENABLED** (required to reach the task).
   - **Security group:** create one with an **inbound rule** — type **Custom TCP**, port **5000**,
     source **Anywhere (0.0.0.0/0)** for a short-lived test (restrict to your IP for anything longer).
4. **Run task**, and wait until **Last status = RUNNING** (about a minute).

**D. Find the public address**
1. Open the running **task** → **Networking** → copy the **Public IP**.
2. The API is now reachable at **`http://<PUBLIC_IP>:5000`**.

> The repository [`README.md`](../README.md#deploying-to-aws-ecs-fargate) contains the full
> AWS CLI equivalent (`create-cluster`, `register-task-definition`, `run-task`) if you prefer
> the terminal; the Console wizard performs the same actions.

## 8. Test the deployed API

The requests are identical to the local tests — substitute the task's **public IP** for `localhost`.

```bash
curl http://<PUBLIC_IP>:5000/health
# {"status":"healthy"}

curl -X POST http://<PUBLIC_IP>:5000/predict \
  -H "Content-Type: application/json" \
  -d '{"make":"MARUTI","model":"SWIFT VDI","age":3,"km_driven":45000}'
# {"predicted_price_lakhs":6.5, ... "label":"Medium" ...}
```

**Using Postman:** method **POST**, URL `http://<PUBLIC_IP>:5000/predict`, header
`Content-Type: application/json`, body (raw → JSON)
`{"make":"MARUTI","model":"SWIFT VDI","age":3,"km_driven":45000}` → **Send**.

> As locally, opening `http://<PUBLIC_IP>:5000/predict` in a browser returns **405** (browsers
> send `GET`). The root path `http://<PUBLIC_IP>:5000/` returns service/usage information.

## 9. Remove all resources (cost cleanup)

A deployment creates several linked resources. Deleting only the obvious one leaves others
that continue to incur cost or clutter the account. This section removes **every** resource,
in dependency order, and then verifies nothing remains.

### What was created, and what incurs cost

| Resource | Created when | Incurs cost? | Action |
| :------- | :----------- | :----------- | :----- |
| **ECS task** (Fargate) | you ran the task | **Yes** — billed per second while RUNNING | Stop it |
| **ECS service** (if created) | optional | **Yes** — it relaunches tasks | Delete it |
| **ECS cluster** | you created it | No (when empty) | Delete it |
| **Task definition** | you registered it | No | Deregister (optional) |
| **CloudWatch log group** `/ecs/car-price-api` | logging enabled | Minor (log storage) | Delete it |
| **ECR repository + image** | you pushed | Minor (image storage) | Delete it |
| **Security group** (custom, port 5000) | you created it | No | Delete it (after the task stops) |
| **IAM role** `ecsTaskExecutionRole` | first ECS use | No (reusable) | Leave it |
| **Default VPC / subnets** | pre-existing | No | Do not delete |

> **In short:** the item that costs money while running is the **Fargate task/service**; the
> items that cost a little while stored are the **CloudWatch log group** and the **ECR image**.

### Teardown order (top to bottom — the task must stop first)

**1. Stop the task / delete the service**
- Console: ECS → cluster. For a **Service**: open it → **Update** → desired tasks **0** →
  **Update**, then **Delete**. For a standalone **Task**: **Tasks** tab → select → **Stop**.
- CLI:
  ```bash
  aws ecs update-service --cluster car-price-cluster --service <SERVICE> --desired-count 0 --region <REGION>
  aws ecs delete-service  --cluster car-price-cluster --service <SERVICE> --force --region <REGION>
  aws ecs stop-task       --cluster car-price-cluster --task <TASK_ARN> --region <REGION>
  ```

**2. Delete the cluster**
- Console: ECS → Clusters → `car-price-cluster` → **Delete cluster**.
- CLI: `aws ecs delete-cluster --cluster car-price-cluster --region <REGION>`

**3. Deregister the task definition(s)** *(optional; keeps the list tidy)*
- Console: ECS → Task definitions → `car-price-api` → select revisions → **Actions → Deregister**.
- CLI: `aws ecs deregister-task-definition --task-definition car-price-api:1 --region <REGION>`

**4. Delete the CloudWatch log group** *(easily overlooked — it keeps storing logs)*
- Console: CloudWatch → Log groups → `/ecs/car-price-api` → **Actions → Delete**.
- CLI: `aws logs delete-log-group --log-group-name /ecs/car-price-api --region <REGION>`

**5. Delete the ECR image and repository**
- Console: ECR → `car-price-api` → select the image(s) → **Delete**; then delete the repository.
- CLI: `aws ecr delete-repository --repository-name car-price-api --force --region <REGION>`

**6. Delete the custom security group** *(only after the task has fully stopped)*
- Console: EC2 → Security Groups → the group allowing inbound TCP 5000 → **Actions → Delete**.
  (The `default` group cannot be deleted; leave it.)
- CLI: `aws ec2 delete-security-group --group-id <SG_ID> --region <REGION>`
- If it reports **"in use"**, the task's network interface has not been released yet — wait a
  minute and retry.

**7. Load balancer / target group / Elastic IP** *(only if you created them)*
- The Console flow in section 7 does not create these. If you added an **Application Load
  Balancer**, delete the load balancer **and** its target group (ALBs bill hourly). Release any
  manually allocated **Elastic IP**. Fargate's auto-assigned public IP is released automatically.

**8. Local cleanup** *(optional)*
```bash
docker rm -f car-price-api                        # remove the container if still running
docker image rm car-price-api car-price-api:dev   # reclaim local disk (~1.5 GB)
```

### Verification checklist

- [ ] ECS → Clusters: no `car-price-cluster` (or no running tasks/services)
- [ ] ECS → Task definitions: `car-price-api` deregistered or absent
- [ ] ECR → Repositories: no `car-price-api`
- [ ] CloudWatch → Log groups: no `/ecs/car-price-api`
- [ ] EC2 → Security Groups: the custom port-5000 group is gone
- [ ] EC2 → Load Balancers / Elastic IPs: nothing you created remains

### Recommended: set a billing alarm

- Console: **Billing and Cost Management → Budgets → Create budget → Zero spend budget**
  (or a small monthly cost budget) → enter your email → **Create**.
- Current charges are visible under **Billing → Bills** and in **Cost Explorer** (updated within ~24h).

> **Operating principle:** stop the task, delete the cluster, the ECR repository, and the
> `/ecs/...` log group; verify each service page is empty; and keep a low-threshold billing alarm.

## 10. Troubleshooting and command reference

| Symptom | Cause and resolution |
| :------ | :------------------- |
| `curl: (7) Failed to connect` to the public IP | The security group does not allow inbound TCP 5000, or the task has no public IP. Re-check section 7C. |
| Task moves to **STOPPED** shortly after starting | Wrong CPU architecture — rebuild with `docker build --platform linux/amd64 …`, re-push, and re-run. Review the task's **Logs**. |
| `405 Method Not Allowed` in a browser | Expected — browsers send `GET`; `/predict` requires `POST`. Use `curl -X POST`, Postman, or `requests`. |
| `{"status":"unhealthy"}` / `503` | The `models/` folder is missing from the image. Confirm `COPY models/ ./models/` is in the `Dockerfile` and the folder existed before building. |
| `400 Unknown make/model` | A typo, or a vehicle absent from the training data. Casing is normalised; spelling is not. |
| ECS cannot pull the image | The task definition's **Image URI** is incorrect, or ECR is in a different Region than the cluster. Use one Region throughout. |
| Security group reports **"in use"** on delete | The task's network interface has not released yet. Wait 1–2 minutes after the task stops, then retry. |

### Command reference

```bash
# Local
docker build -t car-price-api .
docker run -d -p 5000:5000 --name car-price-api car-price-api
curl http://localhost:5000/health
docker rm -f car-price-api

# Push to ECR (Console: ECR → repository → "View push commands")
aws ecr get-login-password --region <REGION> | docker login --username AWS --password-stdin <ACCOUNT_ID>.dkr.ecr.<REGION>.amazonaws.com
docker tag car-price-api:latest <ACCOUNT_ID>.dkr.ecr.<REGION>.amazonaws.com/car-price-api:latest
docker push <ACCOUNT_ID>.dkr.ecr.<REGION>.amazonaws.com/car-price-api:latest

# Deploy on ECS Fargate — Console wizard (section 7), then:
curl http://<PUBLIC_IP>:5000/health

# Cleanup (section 9 — remove everything to stop billing)
aws ecs update-service --cluster car-price-cluster --service <SVC> --desired-count 0 --region <REGION>
aws ecs delete-service  --cluster car-price-cluster --service <SVC> --force --region <REGION>
aws ecs stop-task       --cluster car-price-cluster --task <TASK_ARN> --region <REGION>
aws ecs delete-cluster  --cluster car-price-cluster --region <REGION>
aws logs delete-log-group --log-group-name /ecs/car-price-api --region <REGION>
aws ecr delete-repository  --repository-name car-price-api --force --region <REGION>
aws ec2 delete-security-group --group-id <SG_ID> --region <REGION>
```

This completes the workflow: build the image, validate locally, push to ECR, run on ECS
Fargate, test the live endpoint, and remove all resources afterwards.